# 🛡️ Oversight Arena — TRL GRPO Training Notebook

**Meta × PyTorch OpenEnv Hackathon 2026**

This notebook trains a small open-weight overseer LLM (Qwen-2.5-1.5B-Instruct) to detect
malicious peer agents in collaborative coding, using TRL GRPO against the live OpenEnv
Hugging Face Space.

**Live env:** https://anikasoni-oversight-arena.hf.space  
**GitHub:** https://github.com/anikasoni/oversight-arena  
**HF Space:** https://huggingface.co/spaces/anikasoni/oversight_arena

---

### What this notebook does

1. Installs dependencies
2. Verifies the live OpenEnv Space is healthy
3. Runs a **baseline eval** (untrained model, 30 held-out seeds)
4. Trains with **TRL GRPO** against the live environment (96 prompts, curriculum)
5. Runs a **post-training eval** on the same 30 seeds
6. Shows loss/reward curves and F1 comparison

**Runtime:** T4 free tier ~45 min for 0.5B; A100/L4 ~90 min for 1.5B.  
Change `MODEL` below to switch model size.


In [ ]:
# @title 1. Install dependencies
!pip install -q --upgrade trl>=0.12 peft transformers accelerate datasets
!pip install -q openenv-core>=0.2.1
!pip install -q matplotlib requests bitsandbytes

In [ ]:
# @title 2. Clone repo
import os
if not os.path.exists('oversight-arena'):
    !git clone https://github.com/anikasoni/oversight-arena.git
%cd oversight-arena
!pip install -q -e '.[train]'

In [ ]:
# @title 3. Config — change MODEL for faster/slower run
ENV_URL   = 'https://anikasoni-oversight-arena.hf.space'
MODEL     = 'Qwen/Qwen2.5-1.5B-Instruct'  # swap to 0.5B for faster Colab run
N_PROMPTS = 96
EVAL_N    = 30
LR        = 5e-6
EPOCHS    = 2
print(f'Model: {MODEL}')
print(f'Env:   {ENV_URL}')

In [ ]:
# @title 4. Verify live env is healthy
import requests, json

print('=== /health ===')
r = requests.get(f'{ENV_URL}/health', timeout=15)
print(r.json())

print('\n=== /reset (seed=0, difficulty=0.4) ===')
r = requests.post(f'{ENV_URL}/reset',
                  json={'seed': 0, 'difficulty': 0.4}, timeout=30)
payload = r.json()
obs = payload.get('observation', {})
state = payload.get('state', {})
print('workers:         ', obs.get('workers'))
print('malicious_workers:', state.get('malicious_workers'))
print('malicious_tier:  ', state.get('malicious_tier'))
print('diff (first 300):\n', obs.get('focused_patch_diff', '')[:300])

print('\n=== /step flag W3 ===')
r = requests.post(f'{ENV_URL}/step',
                  json={'action': 'flag_worker', 'worker_id': 'W3',
                        'cwe_tag': 'CWE-476', 'reasoning': 'missing null check'}, timeout=30)
print(r.json().get('reward'), r.json().get('done'))

print('\n=== /grader ===')
r = requests.get(f'{ENV_URL}/grader', timeout=15)
g = r.json()
print('f1:', g.get('f1'), '| tp:', g.get('tp'), '| fp:', g.get('fp'))
print('reward:', g.get('reward'), '| guardrails:', g.get('guardrails_triggered'))

In [ ]:
# @title 5. Run training (TRL GRPO)
import subprocess, sys

cmd = [
    sys.executable, 'scripts/train_grpo.py',
    '--env-url', ENV_URL,
    '--model', MODEL,
    '--n-prompts', str(N_PROMPTS),
    '--num-generations', '4',
    '--batch-size', '4',
    '--grad-accum', '2',
    '--lr', str(LR),
    '--epochs', str(EPOCHS),
    '--curriculum',
    '--eval-after-train',
    '--eval-n', str(EVAL_N),
    '--eval-difficulties', '0.2,0.4,0.6',
    '--eval-samples', '4',
    '--eval-temperature', '0.7',
]

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
print('\nReturn code:', result.returncode)

In [ ]:
# @title 6. Show training results
import json, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

# Training summary
summary_path = Path('results/training_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('=== Training Summary ===')
    print(f"  Model:           {summary.get('model')}")
    print(f"  N prompts:       {summary.get('n_prompts')}")
    print(f"  Mean reward:     {summary.get('mean_reward', 0):.4f}")
    print(f"  First window:    {summary.get('first_window_mean_reward', 0):.4f}")
    print(f"  Last window:     {summary.get('last_window_mean_reward', 0):.4f}")
    print()
    b = summary.get('baseline_eval', {})
    t = summary.get('trained_eval', {})
    print(f"  Baseline F1:     {b.get('mean_f1', 0):.4f}")
    print(f"  Trained F1:      {t.get('mean_f1', 0):.4f}")
    print(f"  Delta F1:        {summary.get('delta_f1', 0):+.4f}")
    print(f"  Delta Reward:    {summary.get('delta_reward', 0):+.4f}")

# Show plots
for png in ['results/loss_curve.png', 'results/reward_curve.png',
            'results/eval_comparison.png', 'results/final_comparison.png']:
    p = Path(png)
    if p.exists():
        print(f'\n--- {p.name} ---')
        display(Image(str(p)))
    else:
        print(f'[missing] {png}')

In [ ]:
# @title 7. Action distribution shift (key evidence)
import csv, collections

for label, path in [('baseline', 'results/eval_baseline_d0.4.csv'),
                    ('trained',  'results/eval_grpo_d0.4.csv')]:
    try:
        rows = list(csv.DictReader(open(path)))
        actions = collections.Counter(r['action'] for r in rows)
        f1s = [float(r['f1']) for r in rows]
        print(f'{label}:')
        print(f'  actions  = {dict(actions)}')
        print(f'  mean_f1  = {sum(f1s)/len(f1s):.3f}')
    except FileNotFoundError:
        print(f'{label}: file not found ({path})')